In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

/Users/sahilnagpal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sahilnagpal/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    What are the 3 intermediate books to learn {programming_language} ?
    Answer by listing them in order of importance.
    '''
)

chat_template_project = ChatPromptTemplate.from_template(
    '''
    What are the 3 projects to learn {programming_language} ?
    Answer by listing them in order of importance in industry.
    '''
)

chat_template_complete = ChatPromptTemplate.from_template(
    """
    I am an intermediate level programmer.

    Consider the following literature:
    {books}

    Also, consider the following projects:
    {projects}

    How much time do you think it will take me to learn ?
    """
)

In [4]:
chat = ChatOpenAI(model_name = 'gpt-4',
                  model_kwargs = {'seed':365},
                  temperature = 0,
                  max_tokens = 500,
                  streaming=True)

/Users/sahilnagpal/Library/Python/3.9/lib/python/site-packages/IPython/core/interactiveshell.py:3490: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [5]:
chain_books = chat_template_books | chat

In [6]:
print(chain_books.invoke({"programming_language": "Python"}).content)

1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho
2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones
3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin


In [7]:
chain_project = chat_template_project | chat

In [8]:
print(chain_project.invoke({"programming_language": "Python"}).content)

1. Data Analysis Project: This involves using libraries like Pandas, NumPy, and Matplotlib to analyze and interpret complex datasets. This project is highly valued in industries like finance, marketing, and tech.

2. Web Scraping Project: This involves using libraries like BeautifulSoup and Scrapy to extract data from websites. This project is important in industries like e-commerce, real estate, and job boards where there's a need to extract and analyze data from various web sources.

3. Web Development Project: This involves using frameworks like Django or Flask to build a website. This project is crucial in the tech industry, especially in roles related to backend development or full-stack development.


In [9]:
chain_parallel = RunnableParallel(
    {
        "books": chain_books,
        "projects": chain_project
    }
)

In [10]:
print(chain_parallel.invoke({"programming_language": "Python"})['books'].content)

1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho
2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones
3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin


In [11]:
print(chain_parallel.invoke({"programming_language": "Python"})['projects'].content)

1. Data Analysis Project: This is the most important project as Python is heavily used in data analysis across various industries. You can start with a project that involves cleaning, organizing, and visualizing data using libraries like Pandas, NumPy, and Matplotlib.

2. Web Development Project: Python is also widely used in web development. A project involving building a simple website using a Python framework like Django or Flask can be a great learning experience.

3. Machine Learning Project: Python is the go-to language for many machine learning projects. A project involving building and training a simple machine learning model using libraries like scikit-learn or TensorFlow can provide valuable experience.


In [12]:
chain_time = (RunnableParallel(
    {
        "books": chain_books,
        "projects": chain_project
    }
)
              | chat_template_complete
              | chat)

In [13]:
print(chain_time.invoke({'programming_language': 'Python'}).content)

The time it takes to learn Python or any programming language can vary greatly depending on several factors such as your current programming knowledge, the amount of time you can dedicate to learning each day, and the complexity of the projects you plan to work on.

If you're an intermediate level programmer, you might already have a good understanding of programming concepts, which can speed up the learning process. 

Reading and understanding the books you mentioned might take you around 1-2 months, assuming you spend a few hours on it every day. 

As for the projects, each one could take anywhere from a few weeks to a few months, depending on the complexity of the project and your familiarity with the libraries and frameworks mentioned. 

So, a rough estimate could be anywhere from 3 to 6 months to get comfortable with Python, its popular libraries, and frameworks. However, remember that everyone learns at their own pace, and the key is consistent practice and application of what yo

In [14]:
chain_time.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+*                     +------------+     
                   ***               ***                   
                      ***         ***   